# Performance benchmarking the new HF Dataset class
We've now made about half of teh Beans0 dataset. Let's see how fast it is to load a dataset,
read single row, and apply a batch processing operation while streaming

In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from cloudpathlib import GSPath, AnyPath
from gcsfs import GCSFileSystem
import os
from dotenv import load_dotenv
load_dotenv()


In [5]:
storage_options = {"project": os.getenv("GCP_DEFAULT_PROJECT")}

# Load and concatenate from storage bucket
uses load_from_disk method, as shown here https://huggingface.co/docs/datasets/filesystems#load-serialized-datasets

In [ ]:
from esp_data.dataset.hf import build_concatenated_dataset, HFDataset
from esp_data.config.db_config import DatasetConfig
from esp_data.file_io import Bucket

In [9]:
p = Bucket("gs://foundation-model-data/beans0/0.0.1")

In [11]:
dirs = p.list_dirs(recursive=False)

In [12]:
cfg = DatasetConfig(
    name="Beans0",
    creator="M. Hagiwara",
    version="0.0.1",
    description="this is the beans0 dataset",
    sources=["esc50", "cbi", "watkins", "hiceas", "humbugdb", "watkins", "enabirds", "rfcx", "gibbons",],
    license="each dataset has a separate license"
)

In [13]:
%%time
dataset = build_concatenated_dataset(dirs, storage_options, sharded=False, new_dataset_config=cfg, version_update_mode=None)

Loading dataset from disk:   0%|          | 0/25 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset from disk:   0%|          | 0/25 [00:00<?, ?it/s]

CPU times: user 4min 20s, sys: 3min 14s, total: 7min 34s
Wall time: 11min 35s


### So it takes 12 min approx to load a memory-mapped 70 Gb dataset

In [ ]:
len(dataset)

In [15]:
dataset.ds

Dataset({
    features: ['source_dataset', 'metadata', 'id', 'created_at', 'derived_from', 'license', 'version', 'path', 'prompt', 'audio', 'text', 'label'],
    num_rows: 42668
})

In [18]:
x = dataset[-1]

In [19]:
x["source_dataset"]

'cbi'

In [20]:
x["metadata"]

{}

In [21]:
len(x["audio"])

441000

# Load and concatenate from disk using data_files approach
This approach is under development. There is much less loading time while streaming

In [6]:
from esp_data.dataset.hf import HFStreamingDataset
from esp_data.config.db_config import DatasetConfig
from esp_data.file_io import Bucket, File

In [7]:
p = Bucket("gs://foundation-model-data/beans0/0.0.1")

In [9]:
# make a fake dataset config
f = File("gs://foundation-model-data/beans0/0.0.1/esc50/dataset_config.json")

In [10]:
f.copy_to(destination="gs://foundation-model-data/beans0/0.0.1/")

In [ ]:
ds = HFStreamingDataset.from_path(str(p), storage_options=storage_options)

# Can we load a single dataset with load_dataset ?

In [26]:
from datasets import load_dataset, load_dataset_builder, load_from_disk

In [27]:
p1 = Bucket(p.bucket_path / "gibbons")
p1

Bucket(gs://foundation-model-data/beans0/0.0.1/gibbons)

THIS DOES NOT WORK
%%time
dss = load_dataset(path="arrow", data_dir=str(p1), storage_options=storage_options, streaming=True)

THIS ALSO DOES NOT WORK
%%time
dss = load_dataset(path=str(p1), storage_options=storage_options, streaming=False)

ANSWER IS NO!

### another try with this:
https://huggingface.co/docs/datasets/loading#arrow

In [31]:
files = list(p1.find_paths_with_extension(".arrow"))

In [32]:
files

[GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00000-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00001-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00002-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00003-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00004-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00005-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00006-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00007-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00008-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00009-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbons/data-00010-of-00012.arrow'),
 GSPath('gs://foundation-model-data/beans0/0.0.1/gibbo

In [34]:

%%time
ds = load_dataset("arrow", data_files=[str(f) for f in files],
                  storage_options=storage_options, streaming=True)

CPU times: user 154 ms, sys: 39.9 ms, total: 194 ms
Wall time: 3.89 s


IterableDatasetDict({
    train: IterableDataset({
        features: ['source_dataset', 'metadata', 'id', 'created_at', 'derived_from', 'license', 'version', 'path', 'prompt', 'audio', 'text', 'label'],
        num_shards: 12
    })
})

# How long does it take to load a single dataset ?

In [36]:
from esp_data.dataset import HFDataset

In [38]:
%%time
# gibbons is the largest individual component of beans
dss = HFDataset.from_path(p1, storage_options=storage_options)

CPU times: user 17.3 s, sys: 11.3 s, total: 28.6 s
Wall time: 52.5 s


In [39]:
len(dss)

18560

In [54]:
dss.columns

['source_dataset',
 'metadata',
 'id',
 'created_at',
 'derived_from',
 'license',
 'version',
 'path',
 'prompt',
 'audio',
 'text',
 'label']

In [40]:
# get batch
batch = dss[0:2]

In [41]:
type(batch)

dict

In [42]:
batch["source_dataset"]

['gibbons', 'gibbons']

In [50]:
batch["license"]

['CC-BY-NC-SA', 'CC-BY-NC-SA']

In [51]:
batch["prompt"]

['<Audio><AudioHere></Audio> Which of these, if any, are present in the audio recording? Single pulse gibbon call, Multiple pulse gibbon call, Gibbon duet, None.',
 '<Audio><AudioHere></Audio> Which of these, if any, are present in the audio recording? Single pulse gibbon call, Multiple pulse gibbon call, Gibbon duet, None.']

In [52]:
batch["text"]

['None', 'None']

In [53]:
batch["label"]

[None, None]

In [67]:
batch["path"]

['HGSM3SOL_0+1_20160320_054700.156_012.wav',
 'HGSM3SOL_0+1_20160320_054700.156_013.wav']

# How long does it take to run a map operation ?

In [23]:
%%time
# FILTER TAKES TOO LONG
sub_ds = dataset.filter(lambda x: x["source_dataset"] == "gibbons", version_update_mode=None, change_log=None)


Filter:   0%|          | 0/42668 [00:00<?, ? examples/s]

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x739b59e21ba0>>
Traceback (most recent call last):
  File "/home/gagan/esp_projects/esp-data/.venv/lib/python3.10/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


CPU times: user 53min 49s, sys: 4min 10s, total: 58min
Wall time: 57min 49s


## Lets try running a map operation now

In [13]:
# get all component dataset json
p = AnyPath("../scripts/temp_cache/")
json_paths = [p / "dataset_config.json" for p in p.iterdir()]
json_paths

[PosixPath('../scripts/temp_cache/hiceas/dataset_config.json'),
 PosixPath('../scripts/temp_cache/cbi/dataset_config.json'),
 PosixPath('../scripts/temp_cache/rfcx/dataset_config.json'),
 PosixPath('../scripts/temp_cache/humbugdb/dataset_config.json'),
 PosixPath('../scripts/temp_cache/enabirds/dataset_config.json'),
 PosixPath('../scripts/temp_cache/gibbons/dataset_config.json'),
 PosixPath('../scripts/temp_cache/esc50/dataset_config.json'),
 PosixPath('../scripts/temp_cache/watkins/dataset_config.json')]

In [14]:
cfgs = {k: None for k in ["esc50", "cbi", "watkins", "hiceas", "humbugdb", "watkins", "enabirds", "rfcx", "gibbons"]}
for path in json_paths:
    c = DatasetConfig.from_json(path)
    cfgs[c.name] = c

In [15]:
cfgs["esc50"].description

'This is a component dataset of the BEANS0 eval dataset.\n            It has following metadata:\n\n\n            {\n    "name": "esc50",\n    "type": "classification",\n    "train_data": "data/esc50/meta/esc50.train.csv",\n    "valid_data": "data/esc50/meta/esc50.valid.csv",\n    "test_data": "data/esc50/meta/esc50.test.csv",\n    "num_labels": 50,\n    "labels": [\n        0,\n        1,\n        2,\n        3,\n        4,\n        5,\n        6,\n        7,\n        8,\n        9,\n        10,\n        11,\n        12,\n        13,\n        14,\n        15,\n        16,\n        17,\n        18,\n        19,\n        20,\n        21,\n        22,\n        23,\n        24,\n        25,\n        26,\n        27,\n        28,\n        29,\n        30,\n        31,\n        32,\n        33,\n        34,\n        35,\n        36,\n        37,\n        38,\n        39,\n        40,\n        41,\n        42,\n        43,\n        44,\n        45,\n        46,\n        47,\n        48,\n   

In [16]:
import json

def make_metadata_from_cfg(cfg: dict):
    idx = cfg["description"].find("metadata:")
    s = cfg["description"][idx + len("metadata:"):].strip()
    return json.loads(s)

In [17]:
metadatas =  {k: make_metadata_from_cfg(v.to_dict()) for k, v in cfgs.items()}


In [18]:
def add_metadata(x: dict):

    # we want to add sample_rate and duration to metadata of each sample
    sr = metadatas[x["source_dataset"]]["sample_rate"]
    dur = len(x["audio"]) / sr
    
    x["metadata"] = {"sample_rate": sr, "duration_secs": dur}
    return x

def add_metadata_batched(batch: dict):
    # now batched is actually dict with lists as values corresponding to batch size
    sample_rates = [metadatas[v]["sample_rate"] for v in batch["source_dataset"]]
    durations = [len(v) / sample_rates[i] for i,v in enumerate(batch["audio"])]

    batch["metadata"] = [{"sample_rate": sr, "duration_secs": dur} for (sr, dur) in zip(sample_rates, durations)]
    return batch

In [20]:
from esp_data.config import DataSample
from pydantic import Field
from typing import Optional

class BeansSample(DataSample):
    """Defines the structure of a Beans0 sample.

    Fields inherited from DataSample:
        - source_dataset: str
        - license: str | None
        - metadata: dict | None
        - created_at: datetime
        - id: str
        - derived_from: str | None
    """

    # required
    path: str = Field(description="Audio filename")
    prompt: str = Field(min_length=1, description="Prompt for naturelm")
    audio: list[float] = Field(description="The audio array")

    # optional
    text: Optional[str] = Field(default=None, description="Some text caption")
    label: Optional[str] = Field(default=None, description="Optional label for the audio")

In [22]:
%%time 
new_ds = ds.map(add_metadata, sample_config=BeansSample, version_update_mode="patch",
                     change_log="=> Added sample_rate and duration_secs fields to the metadata of each sample",
                     batched=True, batch_size=1000)

CPU times: user 533 μs, sys: 73 μs, total: 606 μs
Wall time: 613 μs


In [24]:
%%time 
new_ds = ds.ds.map(add_metadata_batched, batched=True, batch_size=1000)

CPU times: user 476 μs, sys: 0 ns, total: 476 μs
Wall time: 483 μs


In [30]:
%%time
b = []
for batch in new_ds["train"]:
    b.append(batch)
    break
    

CPU times: user 9min 44s, sys: 18.6 s, total: 10min 2s
Wall time: 11min 30s


In [31]:
b = b[0]
type(b)

dict

In [ ]:
print(list(b.keys()))
print(len(b["metadata"]))


In [33]:
print(b["source_dataset"], batch["metadata"])

cbi {'sample_rate': 16000, 'duration_secs': 27.5625}
